# Demonstrasi: Problem Framing & Data Dictionary — PIF304 Pengantar Data Sains
**Pertemuan 3 — Sub-CPMK S02: Menganalisis perumusan masalah dan kebutuhan data**

Notebook ini dipakai dosen untuk demonstrasi di kelas. Kita **tidak** akan membangun model apa pun hari ini. Fokusnya adalah pekerjaan yang sering dilewati padahal paling menentukan:

1. Memastikan **unit analisis** (satu baris = apa?).
2. Membuat **kerangka data dictionary** secara otomatis dari data.
3. Menemukan bagian yang **tidak bisa dijawab oleh data itu sendiri** (definisi, satuan, arti skala) → harus dilengkapi manusia.
4. Mengubah data dictionary menjadi **aturan validasi** yang bisa dijalankan.
5. Menghubungkan **pertanyaan analitik → kebutuhan data → kriteria keberhasilan**.

**Dataset:** hasil survei Google Form mahasiswa Semester 3 dan 5 (45 responden). Kolom `nama` **sudah dianonimkan** menjadi `Mahasiswa 1 … 45`. Beberapa dari Anda mungkin termasuk respondennya — perlakukan data ini dengan hormat.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
sns.set_style("whitegrid")

## 0. Unggah File Data
Jalankan sel di bawah, klik **Choose Files**, lalu pilih file `Data_Survei_Mahasiswa_Anon_PDS.xlsx` dari LMS.

In [ ]:
try:
    from google.colab import files
    print("Pilih file Excel survei (Data_Survei_Mahasiswa_Anon_PDS.xlsx)")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih file-nya.")
    nama_file = list(uploaded.keys())[0]
except ImportError:
    # Di luar Colab (Jupyter lokal): letakkan file di folder yang sama dengan notebook
    nama_file = "Data_Survei_Mahasiswa_Anon_PDS.xlsx"

df = pd.read_excel(nama_file)
print("File:", nama_file, "| ukuran:", df.shape)
df.head()

## 1. Satu Baris Mewakili Apa? (Unit Analisis)
Sebelum menghitung apa pun, pastikan kita tahu **entitas** di balik setiap baris. Pada survei ini dugaan kita: **1 baris = 1 responden (mahasiswa)**. Dugaan itu harus dibuktikan, bukan diasumsikan.

In [ ]:
print("Jumlah baris            :", len(df))
print("Jumlah 'nama' unik      :", df["nama"].nunique())
print("Baris duplikat persis   :", df.duplicated().sum())
print("Duplikat tanpa kolom nama:", df.drop(columns="nama").duplicated().sum())

**Interpretasi:** jumlah `nama` unik = jumlah baris, dan tidak ada baris yang identik — bahkan setelah kolom nama diabaikan. Jadi **1 baris = 1 pengisian formulir**. Tetapi apakah 1 pengisian = 1 mahasiswa *yang berbeda*? Data ini tidak bisa membuktikannya, karena nama sudah dianonimkan dan tidak ada cap waktu/ID responden. Pertanyaan semacam ini dijawab oleh **metadata** pengumpulan data (mis. pengaturan Google Form "batasi 1 respons"), bukan oleh isi data.

**Ilustrasi granularitas:** data yang sama dapat diringkas ke unit lain. Perhatikan bahwa klaim tentang *kelompok* tidak otomatis berlaku untuk *individu* (ecological fallacy).

In [ ]:
per_semester = (df.groupby("semester")
                  .agg(n_responden=("nama", "count"),
                       rata2_ketertarikan=("tingkat_ketertarikan", "mean"),
                       median_waktu_tempuh=("waktu_tempuh_menit", "median"))
                  .round(2))
per_semester

## 2. Kerangka Data Dictionary Otomatis
Pandas dapat memberi tahu **tipe data, jumlah nilai unik, nilai kosong, dan contoh nilai**. Ini membantu, tetapi **tidak** dapat memberi tahu *arti* kolom, *satuan*, atau *arah skala*.

In [ ]:
def buat_kerangka_kamus(data: pd.DataFrame) -> pd.DataFrame:
    """Membuat kerangka data dictionary dari isi data (bagian yang bisa diotomasi)."""
    baris = []
    for kolom in data.columns:
        s = data[kolom]
        numerik = pd.api.types.is_numeric_dtype(s)
        baris.append({
            "variabel": kolom,
            "dtype_pandas": str(s.dtype),
            "n_unik": s.nunique(),
            "n_kosong": int(s.isna().sum()),
            "min": s.min() if numerik else None,
            "max": s.max() if numerik else None,
            "contoh_nilai": ", ".join(map(str, s.dropna().unique()[:4])),
        })
    return pd.DataFrame(baris)

kerangka = buat_kerangka_kamus(df)
kerangka

**Diskusi:** dari tabel di atas, kolom mana yang **tipe pandas-nya** (`int64`, `float64`, `object/str`) **tidak sama** dengan **skala ukurnya** (nominal, ordinal, interval, rasio — materi P2)?

Contoh: `tingkat_ketertarikan` tersimpan sebagai `int64`, tetapi secara konsep adalah **ordinal (Likert)**; `penggunaan_internet` tersimpan sebagai teks, padahal kategorinya **berurutan**.

## 3. Hal yang Tidak Bisa Dijawab oleh Data Itu Sendiri
### 3a. Satuan `jarak_km`

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.4))
sns.stripplot(x=df["jarak_km"], ax=ax, color="#2A7F8C", size=7, jitter=0.15)
ax.set_xlabel("jarak_km (sesuai isian)")
ax.set_title("Sebaran jarak_km: sebagian besar 0-10, beberapa 500-1000")
plt.tight_layout(); plt.show()

In [ ]:
# Uji kewajaran: kecepatan implisit = jarak / waktu tempuh
cek = df[["nama", "moda_transportasi", "jarak_km", "waktu_tempuh_menit"]].copy()
cek["kecepatan_implisit_kmjam"] = (cek["jarak_km"] / (cek["waktu_tempuh_menit"] / 60)).round(1)
cek.sort_values("kecepatan_implisit_kmjam", ascending=False).head(8)

**Temuan:** empat responden mengisi `jarak_km` = 500 atau 1000, semuanya **jalan kaki** dengan waktu tempuh 8–15 menit → kecepatan implisit ribuan km/jam. Penjelasan paling masuk akal: mereka mengisi dalam **meter**, karena satuan tidak ditegaskan di formulir.

Perhatikan juga baris kelima: sepeda motor 5 km dalam 3 menit (≈100 km/jam) — mungkin saja, tetapi patut dipertanyakan. Data dictionary yang baik memuat **aturan konsistensi antarkolom**, bukan hanya rentang per kolom.

Masalah `jarak_km` **bukan** masalah yang diselesaikan dengan rumus outlier; ini masalah **definisi/metadata**. Data dictionary harus mencatat satuan *dan* aturan kewajaran (mis. jalan kaki ≤ 6 km/jam), serta pertanyaan formulir berikutnya perlu diperbaiki.

Mengapa ini penting untuk analisis? Bandingkan ringkasannya:

In [ ]:
mencurigakan = df["jarak_km"] >= 100
print(f"Rata-rata jarak (semua data)              : {df['jarak_km'].mean():.2f} km")
print(f"Rata-rata jarak (tanpa 4 nilai mencurigakan): {df.loc[~mencurigakan, 'jarak_km'].mean():.2f} km")
print(f"Median jarak (semua data)                 : {df['jarak_km'].median():.2f} km")

> Kita **tidak** mengubah data di sini. Keputusan pembersihan (hapus, konversi ke km, atau tandai) dibahas di **Pertemuan 5**, dan harus dicatat dalam log keputusan.

### 3b. Arti skala dan kelengkapan opsi jawaban

In [ ]:
for kolom in ["tingkat_ketertarikan", "frekuensi_pakai_ai", "penggunaan_internet", "waktu_belajar_favorit"]:
    print(f"--- {kolom} ---")
    print(df[kolom].value_counts().sort_index().to_string())
    print()

**Pertanyaan yang tidak bisa dijawab oleh data:**
- `tingkat_ketertarikan` (1–5): tertarik **pada apa**? Apakah 5 = sangat tertarik? Nilai 1 tidak pernah muncul — karena memang tidak ada yang memilih, atau opsi itu tidak tersedia?
- `frekuensi_pakai_ai`: hanya ada *Kadang-kadang, Sering, Sangat sering* → tidak ada *Tidak pernah/Jarang*. Jika opsi itu memang tidak ada di formulir, hasilnya **condong ke atas secara desain**.
- `penggunaan_internet`: **per hari**? Untuk belajar atau total? Bagaimana responden yang memakai < 2 jam?

Semua ini harus ditanyakan ke **pembuat instrumen** (teks pertanyaan asli di Google Form) dan dicatat di data dictionary — termasuk jika jawabannya **belum diketahui** (tandai sebagai asumsi).

## 4. Melengkapi Data Dictionary (Bagian Manual)
Kolom-kolom berikut diisi oleh manusia. Tanda `*` = **asumsi** yang perlu dikonfirmasi ke formulir asli.

In [ ]:
kamus_manual = pd.DataFrame([
    # variabel, definisi, skala, satuan, nilai_valid, catatan
    ["nama", "Pengenal responden (sudah dianonimkan)", "Nominal (ID)", "-", "Mahasiswa 1..45", "Data pribadi asli sudah dihapus; jangan digabung ke data lain"],
    ["usia", "Usia responden saat mengisi survei*", "Rasio (diskret)", "tahun", "16-60", "Kuasi-identitas bila digabung dengan semester & jarak"],
    ["semester", "Semester yang sedang ditempuh", "Ordinal", "-", "Semester 3; Semester 5", "Dapat dikodekan menjadi 3, 5"],
    ["perangkat_utama", "Perangkat utama untuk belajar*", "Nominal", "-", "Smartphone; Laptop; Lainnya", "'Lainnya' tidak dirinci"],
    ["moda_transportasi", "Moda utama ke kampus", "Nominal", "-", "Jalan kaki; Sepeda motor; Ojek online", "Opsi lain di formulir belum diketahui*"],
    ["jarak_km", "Jarak tempat tinggal ke kampus", "Rasio (kontinu)", "kilometer", "0-50", "4 nilai 500/1000 diduga meter; perlu verifikasi"],
    ["waktu_tempuh_menit", "Waktu tempuh sekali jalan ke kampus*", "Rasio (diskret)", "menit", "1-180", "Konsisten dengan moda & jarak?"],
    ["penggunaan_internet", "Lama penggunaan internet per hari*", "Ordinal", "jam/hari*", "2–4 jam < 5–7 jam < Lebih dari 7 jam", "Tidak ada opsi < 2 jam"],
    ["platform_belajar", "Platform belajar yang paling sering dipakai*", "Nominal", "-", "ChatGPT atau AI lainnya; Google/Search Engine; Media sosial; YouTube; Website/Blog; Lainnya", "Pilihan tunggal? perlu dicek"],
    ["frekuensi_pakai_ai", "Frekuensi memakai AI untuk belajar*", "Ordinal", "-", "Kadang-kadang < Sering < Sangat sering", "Skala tidak lengkap (tanpa Jarang/Tidak pernah)"],
    ["waktu_belajar_favorit", "Waktu belajar yang paling disukai", "Nominal", "-", "Pagi; Siang; Malam", "Tidak ada opsi Sore"],
    ["tingkat_ketertarikan", "Ketertarikan terhadap Data Sains*", "Ordinal (Likert)", "skor", "1-5 (5 = paling tertarik*)", "Arah & objek skala perlu dikonfirmasi"],
], columns=["variabel", "definisi", "skala_ukur", "satuan", "nilai_valid", "catatan"])

kamus = kerangka.merge(kamus_manual, on="variabel", how="left")
kamus[["variabel", "definisi", "dtype_pandas", "skala_ukur", "satuan", "nilai_valid", "n_unik", "catatan"]]

## 5. Data Dictionary → Aturan Validasi
Data dictionary yang baik bisa **dijalankan**. Setiap `nilai_valid` dapat diubah menjadi aturan pemeriksaan.

In [ ]:
aturan = {
    "usia":               lambda s: s.between(16, 60),
    "jarak_km":           lambda s: s.between(0, 50),
    "waktu_tempuh_menit": lambda s: s.between(1, 180),
    "tingkat_ketertarikan": lambda s: s.between(1, 5),
    "semester":           lambda s: s.isin(["Semester 3", "Semester 5"]),
    "waktu_belajar_favorit": lambda s: s.isin(["Pagi", "Siang", "Malam"]),
}

hasil = []
for kolom, fungsi in aturan.items():
    lolos = fungsi(df[kolom])
    hasil.append({"variabel": kolom, "n_melanggar": int((~lolos).sum()),
                  "baris_melanggar": ", ".join(df.loc[~lolos, "nama"].tolist())})
pd.DataFrame(hasil)

Hanya `jarak_km` yang melanggar aturan — tepat pada empat baris yang kita curigai. Aturan ini juga dapat dipakai lagi setiap kali survei diulang (**reproducibility**).

## 6. Dari Pertanyaan ke Kebutuhan Data
**Problem statement (contoh):** *Dosen pengampu perlu menentukan slot waktu konsultasi daring karena kehadiran konsultasi tatap muka rendah. Analisis ini menjawab "waktu belajar apa yang paling banyak dipilih, dan apakah berbeda per semester?" pada unit mahasiswa, dan berhasil jika menghasilkan slot yang mencakup ≥ 60% responden.*

Kriteria keberhasilan ditetapkan **sebelum** melihat hasil:

In [ ]:
KRITERIA_CAKUPAN = 0.60   # ditetapkan sebelum analisis

proporsi = df["waktu_belajar_favorit"].value_counts(normalize=True).round(3)
print(proporsi.to_string(), "\n")
print(pd.crosstab(df["semester"], df["waktu_belajar_favorit"], normalize="index").round(2))

slot_terbaik = proporsi.idxmax()
print(f"\nSlot terbanyak: {slot_terbaik} ({proporsi.max():.0%}) -> kriteria {'TERPENUHI' if proporsi.max() >= KRITERIA_CAKUPAN else 'TIDAK terpenuhi'}")

**Batas interpretasi:** hasil ini hanya berlaku untuk 45 responden yang mengisi survei (kemungkinan *self-selection bias*, P2). "Malam" juga belum spesifik — jam berapa? Itu **kesenjangan data** yang harus ditutup dengan pertanyaan lanjutan, bukan ditebak.

### Pertanyaan yang tidak bisa dijawab dataset ini

In [ ]:
kebutuhan = {
    "Apakah pemakaian AI berkaitan dengan prestasi?": ["frekuensi_pakai_ai", "ipk"],
    "Jam berapa tepatnya slot konsultasi dibuka?":   ["waktu_belajar_favorit", "jam_belajar"],
    "Apakah jarak berkaitan dengan waktu tempuh?":   ["jarak_km", "waktu_tempuh_menit"],
}
for pertanyaan, kolom_dibutuhkan in kebutuhan.items():
    hilang = [k for k in kolom_dibutuhkan if k not in df.columns]
    status = "TERSEDIA" if not hilang else f"KURANG: {hilang}"
    print(f"- {pertanyaan}\n    -> {status}")

**Catatan etika:** menambahkan `ipk` berarti menggabungkan survei ini dengan data akademik (SIAKAD). Itu **di luar tujuan pengumpulan awal** dan memerlukan persetujuan baru (purpose limitation & consent, P2). Pilihan yang bertanggung jawab: ubah pertanyaannya, atau rancang pengumpulan data baru yang transparan.

## 7. Simpan Data Dictionary
Data dictionary adalah **artefak**, bukan sekadar catatan di kepala. Simpan bersama data.

In [ ]:
kamus.to_csv("data_dictionary_survei_P3.csv", index=False)
print("Tersimpan: data_dictionary_survei_P3.csv")
try:
    from google.colab import files
    files.download("data_dictionary_survei_P3.csv")
except ImportError:
    pass

## Ringkasan
| Langkah | Yang kita lakukan | Hasil |
|---|---|---|
| Unit analisis | Cek jumlah baris vs nama unik & duplikasi | 1 baris = 1 responden; beberapa jawaban identik perlu metadata |
| Kerangka kamus | Profil otomatis tipe, unik, kosong, contoh | Bagian yang bisa diotomasi |
| Bagian manual | Definisi, skala, satuan, nilai valid, catatan | Banyak asumsi (*) yang harus dikonfirmasi |
| Aturan validasi | Mengubah nilai valid jadi kode | 4 pelanggaran `jarak_km` |
| Pertanyaan → data | Kriteria keberhasilan ditetapkan di awal | Malam 64% ≥ 60%, dengan batas interpretasi |
| Kesenjangan data | Kolom yang dibutuhkan tetapi tidak ada | IPK & jam belajar tidak tersedia |

**Berikutnya (Pertemuan 4):** statistik deskriptif — memilih ringkasan yang tepat untuk setiap skala ukur yang sudah kita dokumentasikan hari ini.